# 84 · Titanic — Inferencia por lotes y entrega a Kaggle

Aplicar el modelo **champion** al `test.csv` de Kaggle, construyes la tabla de
predicciones con su trazabilidad y exportas el CSV listo para hacer *submit* en la competencia.

### Objetivos de aprendizaje
- Reaplicar el pipeline de *features* al `test.csv` usando los parámetros de imputación guardados.
- Cargar el champion desde Unity Catalog por su **alias** y hacer inferencia por lotes.
- Persistir las predicciones con metadatos (versión, run, fecha).
- Exportar `submission_champion.csv` con el formato exacto de Kaggle.


## 1. Configuración

In [0]:
import mlflow
import os
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

CATALOGO = "big_data_ii_2025"
ESQUEMA = "spark_examples"
RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic"
RUTA_EXPERIMENTO = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic_mlflow"
RUTA_DF_TEMPORAL = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic_mlflow_tmp"

TABLA_BRONZE_TEST = f"{CATALOGO}.{ESQUEMA}.titanic_bronze_test"
TABLA_SILVER_TEST = f"{CATALOGO}.{ESQUEMA}.titanic_silver_test"
TABLA_PARAMS_IMPUT = f"{CATALOGO}.{ESQUEMA}.titanic_parametros_imputacion"
TABLA_PREDICCIONES = f"{CATALOGO}.{ESQUEMA}.titanic_predicciones"
NOMBRE_MODELO_UC = f"{CATALOGO}.{ESQUEMA}.titanic_modelo_supervivencia"
RUTA_SUBMISSION = f"{RUTA_VOLUMEN}/submission_champion.csv"

# MLflow usará este directorio al guardar modelos Spark ML
os.environ["MLFLOW_DFS_TMP"] = RUTA_EXPERIMENTO
os.environ["SPARKML_TEMP_DFS_PATH"]=RUTA_DF_TEMPORAL

mlflow.set_registry_uri("databricks-uc")
print("Modelo champion:", f"models:/{NOMBRE_MODELO_UC}@champion")

## 2. Relee los parámetros de imputación (del train)

No los recalculamos con `test.csv`: eso sería *data leakage*. Los leemos de la tabla que guardó la
parte 82. **Los parámetros de imputación son parte del modelo, no de los datos.**

In [0]:
params = spark.table(TABLA_PARAMS_IMPUT).collect()

medianas_age = {}
mediana_fare_pclass = {}
moda_embarked = "S"
for r in params:
    if r["tipo"] == "age":
        titulo, pclass = r["clave"].split("|")
        medianas_age[(titulo, int(pclass))] = float(r["valor"])
    elif r["tipo"] == "fare":
        mediana_fare_pclass[int(r["clave"])] = float(r["valor"])
    elif r["tipo"] == "embarked_moda_valor":
        moda_embarked = r["clave"]

print("Medianas Age:", medianas_age)
print("Medianas Fare:", mediana_fare_pclass)
print("Moda Embarked:", moda_embarked)

## 3. Aplica la MISMA función de features al test

Redefinimos `construir_features` idéntica a la de la parte 82. Como `test.csv` no trae `Survived`, la
función no la toca; opera solo sobre las columnas de entrada.

In [0]:
TITULOS_PRINCIPALES = ["Mr", "Mrs", "Miss", "Master"]


def construir_features(df, medianas_age, mediana_fare_pclass, moda_embarked):
    d = df.withColumn("TituloCrudo", F.regexp_extract(F.col("Name"), r",\s*([^\.]+)\.", 1))
    d = d.withColumn(
        "Title",
        F.when(F.col("TituloCrudo").isin(["Mlle", "Ms"]), F.lit("Miss"))
         .when(F.col("TituloCrudo") == "Mme", F.lit("Mrs"))
         .when(F.col("TituloCrudo").isin(TITULOS_PRINCIPALES), F.col("TituloCrudo"))
         .otherwise(F.lit("Rare")),
    )
    d = d.withColumn("FamilySize", F.col("SibSp") + F.col("Parch") + F.lit(1))
    d = d.withColumn("IsAlone", (F.col("FamilySize") == 1).cast("int"))

    mediana_age_global = float(sum(medianas_age.values()) / max(len(medianas_age), 1))
    cond = F.col("Age").isNull()
    age_case = F.when(F.lit(False), F.lit(None))
    for (titulo, pclass), mediana in medianas_age.items():
        age_case = age_case.when(
            cond & (F.col("Title") == titulo) & (F.col("Pclass") == pclass),
            F.lit(float(mediana)),
        )
    d = d.withColumn(
        "AgeImputed",
        F.when(F.col("Age").isNotNull(), F.col("Age"))
         .otherwise(F.coalesce(age_case, F.lit(mediana_age_global))),
    )

    fare_case = F.when(F.lit(False), F.lit(None))
    for pclass, mediana in mediana_fare_pclass.items():
        fare_case = fare_case.when(F.col("Pclass") == pclass, F.lit(float(mediana)))
    d = d.withColumn(
        "FareImputed",
        F.when(F.col("Fare").isNotNull(), F.col("Fare")).otherwise(fare_case),
    )

    d = d.withColumn(
        "EmbarkedFill",
        F.when(F.col("Embarked").isNull(), F.lit(moda_embarked)).otherwise(F.col("Embarked")),
    )
    d = d.withColumn(
        "Deck",
        F.when(F.col("Cabin").isNull(), F.lit("U")).otherwise(F.substring(F.col("Cabin"), 1, 1)),
    )
    return d.drop("TituloCrudo")

In [0]:
df_test_bronze = spark.table(TABLA_BRONZE_TEST)
df_test_silver = construir_features(
    df_test_bronze, medianas_age, mediana_fare_pclass, moda_embarked
)

# Verificación: sin nulos en las imputadas.
for col in ["AgeImputed", "FareImputed", "EmbarkedFill"]:
    n = df_test_silver.filter(F.col(col).isNull()).count()
    assert n == 0, f"Quedan {n} nulos en {col} tras imputar el test"

df_test_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER_TEST)
print(f"Silver de test escrito: {spark.table(TABLA_SILVER_TEST).count()} filas")

## 4. Carga el champion y haz inferencia

El champion se referencia por su **alias**: `models:/<catálogo>.<esquema>.<modelo>@champion`. Así, si
mañana promueves otra versión moviendo el alias, este notebook sigue funcionando sin cambios.

### ¿`transform` o `spark_udf`?
Depende del *flavor* del champion:
- Si es un **pipeline de SparkML** (flavor `spark`), se carga con `mlflow.spark.load_model` y se aplica
  con `.transform(...)`. Es el camino natural para el modelo que registramos en la parte 83.
- Si fuera un modelo **sklearn/pyfunc**, se usaría
  `mlflow.pyfunc.spark_udf(spark, uri)` y se aplicaría con `withColumn` sobre un DataFrame de features.

Detectamos el *flavor* del champion y seguimos el camino correcto.

In [0]:
client = MlflowClient()
version_champion = client.get_model_version_by_alias(NOMBRE_MODELO_UC, "champion")
run_id_champion = version_champion.run_id
uri_champion = f"models:/{NOMBRE_MODELO_UC}@champion"
print(f"Champion: versión {version_champion.version}  ·  run_id {run_id_champion}")

# Inspeccionamos los flavors del modelo para decidir el camino.
info = mlflow.models.get_model_info(uri_champion)
flavors = list(info.flavors.keys())
print("Flavors disponibles:", flavors)

In [0]:
if "spark" in flavors:
    # Camino SparkML: transform sobre el DataFrame silver de test.
    modelo = mlflow.spark.load_model(uri_champion)
    pred = modelo.transform(df_test_silver)
    # La columna 'prediction' viene como double; extraemos también la probabilidad de sobrevivir.
    from pyspark.ml.functions import vector_to_array
    pred = pred.withColumn("prob_arr", vector_to_array("probability"))
    df_pred = (
        pred.select(
            F.col("PassengerId"),
            F.col("prediction").cast("int").alias("Survived"),
            F.col("prob_arr")[1].alias("probabilidad"),
        )
    )
else:
    # Camino pyfunc/sklearn: spark_udf sobre las columnas de features.
    predecir = mlflow.pyfunc.spark_udf(spark, uri_champion, result_type="double")
    cols_entrada = [c for c in df_test_silver.columns if c != "PassengerId"]
    pred = df_test_silver.withColumn("Survived", predecir(*[F.col(c) for c in cols_entrada]).cast("int"))
    df_pred = pred.select("PassengerId", "Survived").withColumn("probabilidad", F.lit(None).cast("double"))

df_pred.limit(10).display()

## 5. Persiste las predicciones con trazabilidad

Guardamos, junto a cada predicción, la versión del modelo, el `run_id` de origen y la fecha de
*scoring*. Esto permite auditar más adelante con qué modelo se generó cada entrega.

In [0]:
df_pred_final = (
    df_pred
    .withColumn("model_version", F.lit(str(version_champion.version)))
    .withColumn("run_id", F.lit(run_id_champion))
    .withColumn("fecha_scoring", F.current_timestamp())
)

df_pred_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_PREDICCIONES)
n_pred = spark.table(TABLA_PREDICCIONES).count()
print(f"Predicciones guardadas en {TABLA_PREDICCIONES}: {n_pred} filas")

## 6. Exporta el CSV de entrega para Kaggle

Kaggle exige exactamente dos columnas: `PassengerId` y `Survived`. Escribimos un único CSV (no un
directorio particionado) usando pandas, porque el dataset es diminuto.

In [0]:
pdf_sub = (
    spark.table(TABLA_PREDICCIONES)
    .select("PassengerId", "Survived")
    .orderBy("PassengerId")
    .toPandas()
)
pdf_sub.to_csv(RUTA_SUBMISSION, index=False)

# Verificaciones de formato.
assert len(pdf_sub) == 418, f"La entrega debe tener 418 filas, tiene {len(pdf_sub)}"
assert list(pdf_sub.columns) == ["PassengerId", "Survived"], f"Columnas inesperadas: {list(pdf_sub.columns)}"
assert set(pdf_sub["Survived"].unique()).issubset({0, 1}), "Survived debe ser 0 o 1"
print(f"Entrega escrita en {RUTA_SUBMISSION} con {len(pdf_sub)} filas.")
print(pdf_sub.head())

## 7. Descarga el CSV y haz submit en Kaggle

1. Abre **Catalog → `big_data_ii_2025` → `spark_examples` → `titanic`**.
2. Haz clic en `submission_champion.csv` y pulsa **Download**.
3. En Kaggle, ve a la competencia *Titanic* → **Submit Predictions** → arrastra el CSV → **Submit**.
4. Anota tu *public score*.

## 8. (Opcional) Despliegue como endpoint de Model Serving

Databricks Free Edition permite **un número limitado de endpoints CPU** con *scale to zero* (sin GPU, sin
*provisioned throughput*, sin *batch inference* sobre el endpoint). El camino principal de esta
demostración es la inferencia por lotes que acabas de hacer; el endpoint es una extensión.

**Advertencia técnica:** servir un **pipeline de SparkML** en un endpoint CPU es problemático y pesado.

### Pasos en la interfaz (si tienes un modelo pyfunc/XGBoost registrado)
1. Barra lateral → **Serving** → **Create serving endpoint**.
2. Nombre: `titanic-champion`. *Entity*: el modelo  registrado, versión correspondiente.
3. *Compute*: CPU, tamaño **Small**, **Scale to zero: ON**.
4. Espera el estado **Ready** (varios minutos por el *cold start*).
5. En la pestaña **Query endpoint**, prueba con un JSON `dataframe_split` con las columnas de features.
6. **Elimina el endpoint al terminar** (Serving → endpoint → ⋮ → **Delete**)

## Qué aprendiste en esta parte
- Reaplicaste el pipeline de *features* al `test.csv` con los parámetros de imputación del train.
- Cargaste el champion por su **alias** y realizaste inferencia por lotes según su *flavor*.
- Persististe las predicciones con trazabilidad (versión, run, fecha).
- Exportaste `submission_champion.csv` con el formato exacto de Kaggle y lo validaste.

**Fin de la demostración.** Recorriste el ciclo completo: ingesta → EDA → experimentación con MLflow →
tuning → registro en Unity Catalog → inferencia → entrega. 